# CTD Assignment 7: Introduction to Data Visualization

This notebook uses the Diabetes Health Indicators Dataset from Kaggle to practice Matplotlib and Seaborn visualizations.

## Task 1: Data Understanding

Important columns for this assignment:

- `Diabetes_012`: diabetes status, where 0 means no diabetes, 1 means prediabetes, and 2 means diabetes.
- `Age`: age bucket from 1 to 13, not exact age in years.
- `GenHlth`: general health rating from 1 to 5, where 5 is the worst health.
- `PhysHlth`: number of physically unhealthy days during the past 30 days.

Knowing what these columns mean matters because the visualizations tell a better story when the labels and interpretation match the dataset.

## Task 2: Creating the Notebook and Loading Data

In [ ]:
import os

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
csv_path = None

for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        if "diabetes_012" in filename.lower() and filename.endswith(".csv"):
            csv_path = os.path.join(dirname, filename)
            break
    if csv_path:
        break

if csv_path is None:
    raise FileNotFoundError("Could not find a diabetes_012 CSV file. Make sure the Kaggle dataset is attached to this notebook.")

print(csv_path)
diabetes = pd.read_csv(csv_path)
diabetes.head()

## Task 3: A Histogram for Age Distributions

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(diabetes["Age"], bins=13, color="#4C78A8", edgecolor="black")
plt.title("Age Distribution in the Diabetes Health Indicators Dataset")
plt.xlabel("Age Bucket")
plt.ylabel("Count")
plt.xticks(range(1, 14))
plt.show()

The histogram shows how many people fall into each age bucket. Since `Age` is grouped from 1 to 13, the x-axis represents age categories rather than exact ages.

## Task 4: General Health over Time

In [ ]:
health_by_age = diabetes.groupby("Age")[["GenHlth"]].mean()
health_by_age["Health"] = 5 - health_by_age["GenHlth"]
health_by_age = health_by_age.sort_index()

health_by_age.head()

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(health_by_age.index, health_by_age["Health"], marker="o", color="#F58518", linewidth=2)
plt.title("Average Health Score by Age Bucket")
plt.xlabel("Age Bucket")
plt.ylabel("Health Score: Higher Means Better Health")
plt.xticks(range(1, 14))
plt.grid(True, alpha=0.3)
plt.show()

This line plot shows that average health tends to decline as age bucket increases. The story is easy to see because the adjusted `Health` score makes higher values mean better health.

## Task 5: A Heat Map of All Columns

In [ ]:
diabetes_corr = diabetes.corr(numeric_only=True)

plt.figure(figsize=(16, 12))
sns.heatmap(diabetes_corr, cmap="coolwarm", center=0)
plt.title("Correlation Heatmap for All Diabetes Dataset Columns")
plt.show()

This full heatmap includes every numeric column, but it is crowded. The next task makes the correlations easier to understand by focusing on selected columns.

## Task 6: Subset Heat Maps

In [ ]:
important_columns = ["Diabetes_012", "HeartDiseaseorAttack", "GenHlth"]
diabetes_corr_subset = diabetes_corr[important_columns]

diabetes_corr_subset.head()

In [ ]:
diabetes_sorted = diabetes_corr_subset.sort_values("Diabetes_012", ascending=False)

plt.figure(figsize=(8, 7))
sns.heatmap(diabetes_sorted.head(10), annot=True, cmap="coolwarm", center=0, fmt=".2f")
plt.title("Top 10 Positive Correlations with Diabetes")
plt.show()

In [ ]:
plt.figure(figsize=(8, 7))
sns.heatmap(diabetes_sorted.tail(10), annot=True, cmap="coolwarm", center=0, fmt=".2f")
plt.title("Bottom 10 Correlations with Diabetes")
plt.show()

In [ ]:
health_sorted = diabetes_corr_subset.sort_values("GenHlth", ascending=False)

plt.figure(figsize=(8, 7))
sns.heatmap(health_sorted.head(10), annot=True, cmap="coolwarm", center=0, fmt=".2f")
plt.title("Top 10 Positive Correlations with Poor General Health")
plt.show()

In [ ]:
plt.figure(figsize=(8, 7))
sns.heatmap(health_sorted.tail(10), annot=True, cmap="coolwarm", center=0, fmt=".2f")
plt.title("Bottom 10 Correlations with Poor General Health")
plt.show()

The strongest positive correlations with `Diabetes_012` show which factors tend to increase as diabetes status increases. In this dataset, diabetes is often more connected with worse general health, high blood pressure, high cholesterol, BMI, and heart disease or attack than with lifestyle columns alone. For `GenHlth`, the strongest positive relationships describe factors connected with worse self-reported health, while negative correlations such as income or education suggest that people with higher values in those columns may report better general health.

These are correlations, not proof that one variable directly causes another. They are still useful because they show which variables deserve closer attention.

## Task 7: A Pair Plot: Body Mass Index vs. Age

In [ ]:
sample_size = min(5000, len(diabetes))
diabetes_pair_sample = diabetes.sample(n=sample_size, random_state=42)

pair_plot = sns.pairplot(
    diabetes_pair_sample,
    vars=["BMI", "Age"],
    hue="Diabetes_012",
    palette=["#FF5733", "#33FF57", "#3357FF"],
    plot_kws={"alpha": 0.25, "s": 12}
)
pair_plot.fig.suptitle("Pair Plot of BMI and Age by Diabetes Status", y=1.03)
plt.show()

In [ ]:
diabetes["BMI_Quantile"] = pd.qcut(diabetes["BMI"], q=10, labels=False, duplicates="drop") + 1
diabetes_pair_sample = diabetes.sample(n=sample_size, random_state=42)

pair_plot = sns.pairplot(
    diabetes_pair_sample,
    vars=["BMI_Quantile", "Age"],
    hue="Diabetes_012",
    palette=["#FF5733", "#33FF57", "#3357FF"],
    plot_kws={"alpha": 0.25, "s": 12}
)
pair_plot.fig.suptitle("Pair Plot of BMI Quantiles and Age by Diabetes Status", y=1.03)
plt.show()

The first pair plot is difficult to read because BMI has many possible values and many points overlap. The second pair plot groups BMI into 10 quantiles, which makes the relationship between BMI level, age bucket, and diabetes status easier to compare.

## Task 8: Capstone Project Visualization Reminder

For the capstone project notebook, add at least three visualizations. A strong set would include:

- A histogram or box plot to show the distribution of an important numeric column.
- A bar chart to compare categories.
- A scatter plot or heatmap to show relationships between variables.

Each chart should have a title, axis labels, and a markdown explanation of the insight it shows. The explanation should connect the chart back to the main question or purpose of the capstone project.